# M9.1 · Device Behavior Clustering (CRISP-DM Phase 4)

**Goal.** Discover device-behavior **archetypes** without labels.
Project success criterion (clustering half): *"clusters reveal at least 3 distinct driver behavior profiles."*

**Scope decisions enforced here (from EDA verdict 2026-04-30):**
- **F3** — Restrict to `year_month >= '2025-01'`. Pre-2025 the archive-derived signals (harsh events, telemetry, RPM, idle) are effectively absent (~99% of those rows are 2025+).
- **F4** — **Per-tenant** clustering. Tenants 264 (overspeed-rich) and 1787 (harsh-rich) are different operating regimes; a single global cluster space averages them out and produces meaningless centroids.
- **F5** — Unit of analysis is the **device-month**, not the driver. Driver attribution covers 12 of 633 devices in source.
- **F6** — Unsupervised only. The `>0.7` correlation hypothesis is contradicted by the data (harsh ↔ overspeed are inversely correlated globally). We use clustering + anomaly detection (notebook 02).

**Pipeline:**
1. Pull `v_ml_features_full` filtered to >= 2025-01 with full signal.
2. Per tenant: scale → PCA(2) → KMeans(k=3..6) → silhouette → pick best k.
3. Cluster-profile table + 2D scatter plot per tenant.
4. Persist labelled rows to `data/ml/device_clusters.parquet`.

In [ ]:
from __future__ import annotations
import sys, pathlib
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for c in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = c / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src)); break

import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
from accent_fleet.db import get_engine
from sqlalchemy import text

## 2. Inputs

**Modeling window: 2025-01 onward** (F3). **Active rows only:** `total_distance_km >= 100` and `total_ignition_on_minutes > 0` (remove device-months that didn't move — they would form a degenerate "inactive" cluster that swamps the silhouette).

In [ ]:
FEATURES = [
    'overspeed_per_100km', 'avg_speed_over_limit', 'high_speed_trip_ratio',
    'speed_alert_per_100km',
    'harsh_brake_per_100km', 'harsh_accel_per_100km', 'harsh_corner_per_100km',
    'monthly_idle_ratio', 'high_rpm_minutes_per_day',
    'night_trip_ratio', 'rush_hour_trip_ratio',
    'stddev_trip_distance', 'short_trip_ratio',
]
ID_COLS = ['tenant_id', 'device_id', 'year_month']

with get_engine().connect() as conn:
    df = pd.read_sql(text('''
        SELECT * FROM marts.v_ml_features_full
        WHERE year_month >= '2025-01'
          AND total_distance_km >= 100
          AND total_ignition_on_minutes > 0
    '''), conn)
print('rows:', len(df), 'tenants:', sorted(df.tenant_id.unique().tolist()))
print('rows per tenant:'); print(df.tenant_id.value_counts())

## 3. Per-tenant K-Means with silhouette-driven k selection (F4)

In [ ]:
def fit_one_tenant(sub: pd.DataFrame, k_range=range(3, 7)):
    """Scale -> PCA(2 for plotting only) -> KMeans, choosing k by silhouette.
    Clustering itself runs in the original (scaled) feature space, not on PCA."""
    X = sub[FEATURES].fillna(0).to_numpy()
    if len(X) < 50:
        return None
    Xs = StandardScaler().fit_transform(X)
    best = None
    for k in k_range:
        if k >= len(X):
            continue
        km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xs)
        sil = silhouette_score(Xs, km.labels_)
        if best is None or sil > best['sil']:
            best = {'k': k, 'sil': sil, 'km': km, 'Xs': Xs}
    pca2 = PCA(n_components=2, random_state=42).fit_transform(best['Xs'])
    out = sub[ID_COLS].copy()
    out['cluster'] = best['km'].labels_
    out['pc1'] = pca2[:, 0]; out['pc2'] = pca2[:, 1]
    return {'best_k': best['k'], 'silhouette': best['sil'],
            'labels': out, 'centroids': best['km'].cluster_centers_}

results = {}
for tenant_id, sub in df.groupby('tenant_id'):
    r = fit_one_tenant(sub)
    if r is None:
        print(f'tenant {tenant_id}: only {len(sub)} rows — skipped'); continue
    results[tenant_id] = r
    print(f'tenant {tenant_id}: n={len(sub):>4d}  best_k={r["best_k"]}  silhouette={r["silhouette"]:.3f}')

## 4. Cluster profiles (per tenant)

For each tenant, show the per-cluster **mean** of every feature so we can label the clusters semantically (e.g., "aggressive speeders", "efficient drivers", "high-idle commuters").

In [ ]:
all_labels = []
for tenant_id, r in results.items():
    labels = r['labels'].merge(df[ID_COLS + FEATURES + ['total_distance_km']],
                                on=ID_COLS, how='left')
    all_labels.append(labels)
    profile = (labels.groupby('cluster')[FEATURES + ['total_distance_km']]
                      .mean().round(2))
    profile['n'] = labels.groupby('cluster').size()
    print(f'\n=== tenant {tenant_id} (k={r["best_k"]}, sil={r["silhouette"]:.3f}) ===')
    display(profile)
all_labels = pd.concat(all_labels, ignore_index=True)

## 5. 2D scatter (PCA) per tenant

In [ ]:
n = len(results)
if n:
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4), squeeze=False)
    for ax, (tenant_id, r) in zip(axes[0], results.items()):
        L = r['labels']
        ax.scatter(L.pc1, L.pc2, c=L.cluster, cmap='tab10', s=10, alpha=0.7)
        ax.set_title(f'tenant {tenant_id}  k={r["best_k"]}  sil={r["silhouette"]:.2f}')
        ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    plt.tight_layout(); plt.show()

## 6. Persist labels for downstream use

In [ ]:
out_dir = pathlib.Path('../../data/ml'); out_dir.mkdir(parents=True, exist_ok=True)
all_labels[ID_COLS + ['cluster']].to_parquet(out_dir / 'device_clusters.parquet', index=False)
print('wrote', out_dir / 'device_clusters.parquet', 'rows:', len(all_labels))

## 7. Exit gate

Project success criterion: *≥3 distinct profiles*. We pass if at least one tenant's best_k >= 3 AND silhouette >= 0.15 (a low bar appropriate for fleet behavior data).

In [ ]:
passed = [(t, r) for t, r in results.items() if r['best_k'] >= 3 and r['silhouette'] >= 0.15]
print('tenants meeting gate:', [t for t, _ in passed], 'of', list(results.keys()))
assert passed, 'No tenant met the >=3 cluster, >=0.15 silhouette gate.'